# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Each record in the dataset corresponds to the daily performance of a specific content page belonging to a specific client.

For the development and verification of this lane, the analysis will focus on data from March 2026. This period will be used to examine content performance and create the required features for identifying content refresh opportunities.

June 2026 will be reserved as the final test period. Therefore, it will be excluded from development and feature-building activities to prevent information from the test period from influencing the results.


In [1]:
%pip -q install duckdb

import duckdb

con = duckdb.connect()

print("DuckDB is ready.")

DuckDB is ready.


In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("HF_TOKEN found successfully.")
else:
    print("HF_TOKEN was not found. Check Colab Secrets.")

HF_TOKEN found successfully.


In [3]:
import os

os.environ["HF_TOKEN"] = HF_TOKEN

print("Hugging Face token is ready for the warehouse connection.")

Hugging Face token is ready for the warehouse connection.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features

### Candidate Features

I will consider the following observed performance metrics as potential input features:

* `gsc_impressions` — the number of times the page appeared in Google Search results.
* `gsc_clicks` — the number of clicks received from Google Search.
* `gsc_avg_position` — the observed average position of the page in search results.
* `ga4_sessions` — the number of GA4 sessions, provided that GA4 data is available.
* **CTR** — a derived metric calculated as `gsc_clicks / gsc_impressions` when the number of impressions is greater than zero.

These variables represent performance information that has already been observed and can therefore be available before making a future content-refresh decision.

### Label

The label will capture a future observed performance outcome. It will be used to determine the content refresh opportunity for a page.

Information from this future outcome must not be used as an input feature, since doing so would introduce data leakage.

### Context Fields

The following columns will be retained as contextual information:

* `client_hash_id` — identifies the client associated with the observation.
* `content_hash_id` — identifies the specific content page.
* `report_date` — specifies the date of the observation.
* `month` — indicates the month from which the data was collected.

These fields are useful for identifying, grouping, joining, and separating observations, but they will not be used as predictive features.

### Excluded Fields and Data

The following information will be left out of the model inputs:

* **Future performance data**, as it would introduce information from the future and cause leakage.
* **Fields derived directly from the label**, because they could reveal the target outcome to the model.
* `client_hash_id` and `content_hash_id`, because they are identifiers rather than actual performance indicators.
* **June 2026 data during development**, since this month is reserved as a sealed test period.

GA4-related metrics will also be handled carefully. The `ga4_data_available` field will be checked so that unavailable GA4 data is not incorrectly interpreted as actual zero performance.

In [4]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ctr"
]

context = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month"
]

excluded = [
    "future performance information",
    "label-derived fields",
    "client_hash_id as a model feature",
    "content_hash_id as a model feature",
    "June 2026 during development"
]

print("Candidate features:", features)
print("Context fields:", context)
print("Excluded:", excluded)

Candidate features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ctr']
Context fields: ['client_hash_id', 'content_hash_id', 'report_date', 'month']
Excluded: ['future performance information', 'label-derived fields', 'client_hash_id as a model feature', 'content_hash_id as a model feature', 'June 2026 during development']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Add it in Colab Secrets.")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("Hugging Face authentication is ready.")

Hugging Face authentication is ready.


In [6]:
schema = con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [7]:
query1 = """
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS number_of_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
"""

result1 = con.sql(query1).df()

print("Query 1: Grain check")
display(result1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1: Grain check


,client_hash_id,content_hash_id,report_date,number_of_rows


In [8]:
query2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

result2 = con.sql(query2).df()

print("Query 2: March 2026 row count and date range")
display(result2)

Query 2: March 2026 row count and date range


,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [9]:
# QUERY 3 — Check GA4 availability

query3 = """
SELECT
    COUNT(*) AS rows_with_ga4_available
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
"""

result3 = con.sql(query3).df()

print("Query 3: Rows where GA4 data is available")
display(result3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3: Rows where GA4 data is available


,rows_with_ga4_available
0,413966


## Five features

For the March 2026 observation period, I will use five features to represent the performance of each content page:

1. **Impressions** — the number of times the page appeared in search results during March. Since these impressions have already been recorded, they are available when the decision is made.

2. **Clicks** — the number of search clicks received by the page during March. This is an observed metric and can therefore be used at the decision point.

3. **Sessions** — the number of sessions recorded through analytics during the observation period. These observations are available before the page-review decision.

4. **CTR** — the click-through rate calculated using the observed impressions and clicks. Since both values are already available, CTR can also be calculated before making the decision.

5. **Average position** — the average search position observed for the page during March. This is also available at the time of the decision.

Together, these five features represent the page's observed search and analytics performance. Only information available from the observation period is used as an input; information about future outcomes is excluded to avoid data leakage.

In [10]:
# Build the five-feature frame

query_features = """
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(ga4_sessions) AS sessions,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS ctr,

    AVG(gsc_avg_position) AS avg_position

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE ga4_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.sql(query_features).df()

print("Five-feature dataframe:")
display(features.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Five-feature dataframe:


,client_hash_id,content_hash_id,impressions,clicks,sessions,ctr,avg_position
0,client_65de48885f4ef01b,content_5e120e972f11f833,0.0,0.0,3.0,NaN,NaN
1,client_65de48885f4ef01b,content_4ab81290aec524dd,0.0,0.0,1.0,NaN,NaN
2,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,458.0,2.0,14.0,0.436681,4.418032
3,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3943.0,23.0,54.0,0.583312,4.392897
4,client_65de48885f4ef01b,content_aba6e5270431d8ef,0.0,0.0,5.0,NaN,NaN
5,client_65de48885f4ef01b,content_3c286ded8bd68120,2180.0,15.0,30.0,0.688073,8.439390
6,client_65de48885f4ef01b,content_9d17d30b63eaa640,0.0,0.0,1.0,NaN,NaN
7,client_65de48885f4ef01b,content_b2108e8fe3360fa6,503.0,8.0,23.0,1.590457,5.531459
8,client_65de48885f4ef01b,content_0535f4407e4320df,0.0,0.0,4.0,NaN,NaN
9,client_65de48885f4ef01b,content_ff867882e604fa96,24.0,0.0,2.0,0.000000,2.850000


DATA LEAKAGE


In [11]:
# FOUR — THE TRAP: DELIBERATE DATA LEAKAGE

from sklearn.metrics import accuracy_score

leak_demo = features.copy()

# Create a simple demonstration label
leak_demo["label"] = (
    leak_demo["clicks"] < leak_demo["impressions"] * 0.01
).astype(int)

# DELIBERATELY LEAK THE LABEL INTO ONE FEATURE
leak_demo["leaky_feature"] = leak_demo["label"]

# With leakage, prediction is exactly the label
leaky_prediction = leak_demo["leaky_feature"]

leaky_score = accuracy_score(
    leak_demo["label"],
    leaky_prediction
)

print("Score WITH deliberate leakage:", leaky_score)

# REMOVE THE LEAKED FEATURE
honest_features = leak_demo.drop(
    columns=["leaky_feature"]
)

print("\nLeaky feature removed.")
print("Remaining columns:")
print(honest_features.columns.tolist())

# Confirm that leakage is gone
assert "leaky_feature" not in honest_features.columns

print("\nPASS: The leaked feature is no longer being used.")

Score WITH deliberate leakage: 1.0

Leaky feature removed.
Remaining columns:
['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'sessions', 'ctr', 'avg_position', 'label']

PASS: The leaked feature is no longer being used.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

There are several limitations that need to be considered when working with this dataset.

First, the amount of historical data is not the same for every client. Some clients may have data available for longer periods than others, which means the observations may not be equally complete across all clients.

Second, the availability of Google Search Console (GSC) and Google Analytics 4 (GA4) data can vary between observations. An observation may contain GSC information while GA4 information is unavailable. Because of this, a missing or zero GA4 value should not automatically be considered evidence of zero user engagement.

Third, when data from different tables or time periods is combined, their observation windows may overlap. Features must therefore be constructed carefully so that they only contain information that was available before the prediction or ranking decision was made. This helps prevent data leakage.

Finally, the dataset contains observational rather than experimental data. It can be used to identify measurable performance patterns and support the ranking of pages for content review, but it cannot establish that refreshing a page will directly cause an increase in traffic or search rankings.

The available data also does not provide direct access to Google's internal ranking system, so it cannot be used to determine or predict the exact algorithm used by Google to rank pages.

In [12]:
print("Data limits recorded:")
print("- Client history is not balanced.")
print("- GA4 availability differs across observations.")
print("- Future information must not be used as a feature.")
print("- The data supports decision-support, not causal claims.")

Data limits recorded:
- Client history is not balanced.
- GA4 availability differs across observations.
- Future information must not be used as a feature.
- The data supports decision-support, not causal claims.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.